In [1]:
from src.data_loader import load_anime_dataset
ratings, anime = load_anime_dataset()

from src.utils import preprocess_data
_, ratings = preprocess_data(ratings, min_likes_user=100, min_likes_anime=50)
print("Ratings shape:", ratings.shape)
print(ratings)

/home/tun/project/anime-recommendation/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ratings shape: (855449, 6)
                user_id  anime_id  score     status  episodes_seen  liked
146             Hamilho     52991   10.0  Completed           28.0      1
197         farhantwoo_     52991   10.0  Completed           28.0      1
213         RiasReverie     52991   10.0  Completed           28.0      1
243           chefconke     52991   10.0  Completed           28.0      1
261           dragonnir     52991   10.0  Completed           28.0      1
...                 ...       ...    ...        ...            ...    ...
16572619     Arashiiiii     51132    3.0  Completed            8.0      0
16572629         DSAMei     51132   10.0  Completed            8.0      1
16572630  Sake_and_Soju     51132    6.0  Completed            8.0      0
16572633   Coolest1234D     51132    2.0  Completed            8.0      0
16572634  FuckingPsycho     51132    6.0  Completed            8.0      0

[855449 rows x 6 columns]


In [2]:
import pandas as pd

# Suppose df is your DataFrame
user2id = {uid: idx for idx, uid in enumerate(ratings['user_id'].unique())}
anime2id = {aid: idx for idx, aid in enumerate(ratings['anime_id'].unique())}

ratings['user_idx'] = ratings['user_id'].map(user2id)
ratings['anime_idx'] = ratings['anime_id'].map(anime2id)
ratings['label'] = ratings['liked']
print(ratings.shape)
print(ratings)

(855449, 9)
                user_id  anime_id  score     status  episodes_seen  liked  \
146             Hamilho     52991   10.0  Completed           28.0      1   
197         farhantwoo_     52991   10.0  Completed           28.0      1   
213         RiasReverie     52991   10.0  Completed           28.0      1   
243           chefconke     52991   10.0  Completed           28.0      1   
261           dragonnir     52991   10.0  Completed           28.0      1   
...                 ...       ...    ...        ...            ...    ...   
16572619     Arashiiiii     51132    3.0  Completed            8.0      0   
16572629         DSAMei     51132   10.0  Completed            8.0      1   
16572630  Sake_and_Soju     51132    6.0  Completed            8.0      0   
16572633   Coolest1234D     51132    2.0  Completed            8.0      0   
16572634  FuckingPsycho     51132    6.0  Completed            8.0      0   

          user_idx  anime_idx  label  
146              0      

In [3]:
import numpy as np

def make_train_test_split_df(df, holdout_ratio=0.1, seed=1234):
    np.random.seed(seed)
    train_indices = []
    test_indices = []

    # Only split on positive (liked) interactions
    users = df['user_idx'].unique()
    for user in users:
        user_likes = df[(df['user_idx'] == user) & (df['label'] == 1)].index.tolist()
        test_size = int(np.floor(len(user_likes) * holdout_ratio))

        test_like_idx = np.random.choice(user_likes, size=test_size, replace=False)
        test_indices.extend(test_like_idx)
        train_indices.extend([i for i in user_likes if i not in test_like_idx])

    train_indices += df[df['label'] == 0].index.tolist()
    
    train_df = df.loc[train_indices].reset_index(drop=True)
    test_df = df.loc[test_indices].reset_index(drop=True)
    return train_df, test_df

train_df, test_df = make_train_test_split_df(ratings, holdout_ratio=0.1, seed=1234)
print(train_df)
print(test_df)

                user_id  anime_id  score     status  episodes_seen  liked  \
0               Hamilho     52991   10.0  Completed           28.0      1   
1               Hamilho     43608   10.0  Completed           13.0      1   
2               Hamilho     28851    8.0  Completed            1.0      1   
3               Hamilho     37510    9.0  Completed           13.0      1   
4               Hamilho     58567   10.0  Completed           13.0      1   
...                 ...       ...    ...        ...            ...    ...   
808218  BalancingFlame4     51132    6.0  Completed            8.0      0   
808219       Arashiiiii     51132    3.0  Completed            8.0      0   
808220    Sake_and_Soju     51132    6.0  Completed            8.0      0   
808221     Coolest1234D     51132    2.0  Completed            8.0      0   
808222    FuckingPsycho     51132    6.0  Completed            8.0      0   

        user_idx  anime_idx  label  
0              0          0      1  
1

In [6]:
from torch.utils.data import Dataset, DataLoader

class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = df['user_idx'].values
        self.items = df['anime_idx'].values
        self.labels = df['label'].values.astype(np.float32)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

train_dataset = RatingsDataset(train_df)
print(train_dataset)
test_dataset = RatingsDataset(test_df)
print(test_dataset)

In [ ]:
import torch
import torch.nn as nn
from src.base_model import BaseRecommender
from src.evaluate import evaluate_at_k

class NCF(nn.Module, BaseRecommender):
    def __init__(self, num_users, num_items, emb_size=32, mlp_layers=[64, 32, 16, 8]):
        super(NCF, self).__init__()
        self.num_users = num_users
        self.num_items = num_items

        # GMF Embeddings
        self.user_emb_gmf = nn.Embedding(num_users, emb_size)
        self.item_emb_gmf = nn.Embedding(num_items, emb_size)
        # MLP Embeddings
        self.user_emb_mlp = nn.Embedding(num_users, emb_size)
        self.item_emb_mlp = nn.Embedding(num_items, emb_size)

        # MLP Layers
        mlp_modules = []
        input_size = emb_size * 2
        for layer_size in mlp_layers:
            mlp_modules.append(nn.Linear(input_size, layer_size))
            mlp_modules.append(nn.ReLU())
            input_size = layer_size
        self.mlp = nn.Sequential(*mlp_modules)

        # Final Prediction
        self.output_layer = nn.Linear(mlp_layers[-1] + emb_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_idx, item_idx):
        # GMF
        gmf_user = self.user_emb_gmf(user_idx)
        gmf_item = self.item_emb_gmf(item_idx)
        gmf_out = gmf_user * gmf_item

        # MLP
        mlp_user = self.user_emb_mlp(user_idx)
        mlp_item = self.item_emb_mlp(item_idx)
        mlp_in = torch.cat([mlp_user, mlp_item], dim=-1)
        mlp_out = self.mlp(mlp_in)

        # NeuMF
        concat = torch.cat([gmf_out, mlp_out], dim=-1)
        out = self.output_layer(concat)
        return self.sigmoid(out).squeeze()

    def fit(self, train_tensor, test_tensor, epochs=20, batch_size=512, learning_rate=0.001, k=10, device='cpu', **kwargs):
        """
        train_tensor: shape (num_interactions, 3) where each row is (user_idx, item_idx, label)
        test_tensor: same format as train_tensor
        """
        self.to(device)
        train_tensor = train_tensor.to(device)
        test_tensor = test_tensor.to(device)

        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate, weight_decay=1e-5)
        criterion = nn.BCELoss()
        losses, precs, maps, ndcgs = [], [], [], []

        best_map = 0.0
        patience = 3
        patience_counter = 0
        best_model_state = None

        for epoch in range(epochs):
            self.train()
            epoch_loss = 0.0
            num_batches = 0

            # Shuffle for each epoch
            idx = torch.randperm(train_tensor.size(0))
            train_tensor = train_tensor[idx]

            for i in range(0, train_tensor.size(0), batch_size):
                batch = train_tensor[i:i+batch_size]
                users = batch[:, 0].long()
                items = batch[:, 1].long()
                labels = batch[:, 2].float()

                preds = self.forward(users, items)
                loss = criterion(preds, labels)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                num_batches += 1

            epoch_loss /= num_batches
            losses.append(epoch_loss)

            # Evaluation
            self.eval()
            with torch.no_grad():
                precision, mean_ap, mean_ndcg = evaluate_at_k(self, train_tensor, test_tensor, k=k, device=device)
                precs.append(precision)
                maps.append(mean_ap)
                ndcgs.append(mean_ndcg)

            if mean_ap > best_map:
                best_map = mean_ap
                patience_counter = 0
                best_model_state = self.state_dict()
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("Early stopping triggered.")
                    break

            print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.4f} | "
                  f"Precision@{k}: {precision:.4f} | MAP@{k}: {mean_ap:.4f} | NDCG@{k}: {mean_ndcg:.4f}")

        if best_model_state is not None:
            self.load_state_dict(best_model_state)

        return {
            "losses": losses,
            "precs": precs,
            "maps": maps,
            "ndcgs": ndcgs
        }

    def predict(self, user_item_tensor, device='cpu'):
        """
        user_item_tensor: shape (num_pairs, 2), where each row is (user_idx, item_idx)
        Returns: predicted scores as a tensor
        """
        self.eval()
        user_item_tensor = user_item_tensor.to(device)
        with torch.no_grad():
            users = user_item_tensor[:, 0].long()
            items = user_item_tensor[:, 1].long()
            preds = self.forward(users, items)
        return preds

    def save(self, path):
        torch.save(self.state_dict(), path)

    def load(self, path, device='cpu'):
        self.load_state_dict(torch.load(path, map_location=device))
        self.eval()

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NCF(num_users=len(user2id), num_items=len(anime2id)).to(device)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop


Epoch 1, Loss: 0.5363
Epoch 2, Loss: 0.4611
Epoch 3, Loss: 0.4374
Epoch 4, Loss: 0.4205
Epoch 5, Loss: 0.4086
Epoch 6, Loss: 0.3993
Epoch 7, Loss: 0.3908
Epoch 8, Loss: 0.3821
Epoch 9, Loss: 0.3736
Epoch 10, Loss: 0.3645
Epoch 11, Loss: 0.3550
Epoch 12, Loss: 0.3451
Epoch 13, Loss: 0.3348
Epoch 14, Loss: 0.3241
Epoch 15, Loss: 0.3131
Epoch 16, Loss: 0.3021
Epoch 17, Loss: 0.2905
Epoch 18, Loss: 0.2785
Epoch 19, Loss: 0.2669
Epoch 20, Loss: 0.2549
Epoch 21, Loss: 0.2429
Epoch 22, Loss: 0.2310
Epoch 23, Loss: 0.2190
Epoch 24, Loss: 0.2071
Epoch 25, Loss: 0.1951
Epoch 26, Loss: 0.1836
Epoch 27, Loss: 0.1721
Epoch 28, Loss: 0.1611
Epoch 29, Loss: 0.1502
Epoch 30, Loss: 0.1396
